# IPCA vs SELIC – Política Monetária, Defasagens e Features para ML

Neste notebook vamos analisar a relação entre:

- IPCA (inflação)
- SELIC (política monetária)

Em três camadas:

1. Estatística descritiva e defasagens
2. Econometria (VAR + IRF)
3. Geração de features para modelos de machine learning

Este arquivo será usado via Jupytext como notebook.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import ccf

from src.database.connection import engine

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

print("PYTHONPATH atualizado:", ROOT)

pd.set_option("display.float_format", "{:.4f}".format)

## 1. Carregar séries de IPCA e SELIC

Vamos usar:
- IPCA mensal (índice cheio)
- SELIC meta (série mensal)

In [ ]:
query_ipca = """
SELECT
    date::date,
    indice AS ipca
FROM bcb.ipca
WHERE date >= '2012-01-01'
ORDER BY date;
"""

df_ipca = pd.read_sql(query_ipca, engine)

query_selic = """
SELECT
    date::date,
    value AS selic
FROM bcb.selic_meta
WHERE date >= '2012-01-01'
ORDER BY date;
"""

df_selic = pd.read_sql(query_selic, engine)

df_ipca.head(), df_selic.head()

## 2. Base consolidada IPCA x SELIC

Unimos as duas séries em uma base única, com frequência mensal.

In [ ]:
df = pd.merge(df_ipca, df_selic, on="date", how="inner")
df = df.set_index("date").asfreq("MS")

df.tail()

## 3. Visão geral – IPCA vs SELIC

Primeiro, um gráfico simples para enxergar a dinâmica conjunta.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df.index, df["ipca"], label="IPCA mensal (%)", color="steelblue")
plt.plot(df.index, df["selic"], label="SELIC meta (%)", color="darkred")
plt.title("IPCA vs SELIC – Visão Geral")
plt.xlabel("Data")
plt.ylabel("Taxa (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 4. Correlação com defasagens (lags)

Vamos calcular a correlação entre:
- IPCA corrente
- SELIC defasada em 0 a 18 meses

Interpretação:
- SELIC defasada → impacto da política monetária sobre a inflação futura.

In [ ]:
max_lag = 18
rows = []

for lag in range(0, max_lag + 1):
    corr = df["selic"].shift(lag).corr(df["ipca"])
    rows.append({"lag_meses": lag, "correlacao": corr})

df_corr = pd.DataFrame(rows)
df_corr

In [ ]:
plt.figure(figsize=(8, 3))
plt.stem(df_corr["lag_meses"], df_corr["correlacao"], use_line_collection=True)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Correlação IPCA vs SELIC defasada")
plt.xlabel("Defasagem da SELIC (meses)")
plt.ylabel("Correlação")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Correlação móvel (rolling correlation)

Aqui olhamos como a correlação entre SELIC e IPCA muda ao longo do tempo.

In [ ]:
df["rolling_corr_12m"] = df["selic"].rolling(12).corr(df["ipca"])

plt.figure(figsize=(10, 3))
plt.plot(df.index, df["rolling_corr_12m"], color="purple")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Correlação móvel (12m) entre SELIC e IPCA")
plt.xlabel("Data")
plt.ylabel("Correlação (12m)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Cross-Correlation Function (CCF)

A CCF ajuda a identificar defasagens em que a SELIC explica melhor o IPCA.
Aqui usamos as séries alinhadas e olhamos os primeiros 18 lags.

In [ ]:
ipca_series = df["ipca"].dropna()
selic_series = df["selic"].dropna()

min_len = min(len(ipca_series), len(selic_series))
ipca_series = ipca_series[-min_len:]
selic_series = selic_series[-min_len:]

cc = ccf(selic_series, ipca_series)
lags = np.arange(len(cc))

cc_df = pd.DataFrame({"lag": lags, "ccf": cc})
cc_df = cc_df.iloc[:19]  # primeiros 18 lags + contemporâneo
cc_df

In [ ]:
plt.figure(figsize=(8, 3))
plt.stem(cc_df["lag"], cc_df["ccf"], use_line_collection=True)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Cross-Correlation Function – SELIC → IPCA")
plt.xlabel("Lag (meses)")
plt.ylabel("CCF")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Gráfico de defasagem – SELIC defasada vs IPCA

Escolhemos um lag representativo (por exemplo, 9 meses) com base no gráfico de correlação.
Ajuste `lag_escolhido` conforme a evidência empírica.

In [ ]:
lag_escolhido = 9

df["selic_lag"] = df["selic"].shift(lag_escolhido)

plt.figure(figsize=(10, 4))
plt.plot(df.index, df["ipca"], label="IPCA mensal (%)", color="steelblue")
plt.plot(
    df.index,
    df["selic_lag"],
    label=f"SELIC defasada {lag_escolhido}m",
    color="darkred",
)
plt.title(f"IPCA vs SELIC defasada em {lag_escolhido} meses")
plt.xlabel("Data")
plt.ylabel("Taxa (%)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 8. Modelo VAR – IPCA e SELIC

Agora estimamos um VAR simples com:
- IPCA
- SELIC

Para analisar a função impulso-resposta (IRF) de um choque na SELIC sobre o IPCA.

In [ ]:
df_var = df[["ipca", "selic"]].dropna()

model = VAR(df_var)
results = model.fit(maxlags=12, ic="aic")
results.summary()

### 8.1. Função Impulso-Resposta (IRF)

Um choque na SELIC e sua resposta estimada no IPCA ao longo de 24 meses.

In [ ]:
irf = results.irf(24)

plt.figure(figsize=(8, 4))
irf.plot(orth=True, impulse="selic", response="ipca")
plt.suptitle("IRF – Choque na SELIC sobre o IPCA", y=1.02)
plt.tight_layout()
plt.show()

### 8.2. Decomposição da variância do erro de previsão (FEVD)

Quanto da variância do IPCA é explicada por choques na SELIC?

In [ ]:
fevd = results.fevd(24)

plt.figure(figsize=(8, 4))
fevd.plot(figsize=(8, 4))
plt.suptitle("FEVD – Decomposição da variância (IPCA e SELIC)", y=1.02)
plt.tight_layout()
plt.show()

## 9. Feature engineering para Machine Learning

Aqui preparamos features derivadas de:
- defasagens da SELIC
- defasagens do IPCA
- juros reais e spreads

Essas features serão usadas em modelos supervisionados (regressão, árvores, etc.).

In [ ]:
# Lags da SELIC e do IPCA
for lag in [1, 3, 6, 9, 12]:
    df[f"selic_lag_{lag}"] = df["selic"].shift(lag)
    df[f"ipca_lag_{lag}"] = df["ipca"].shift(lag)

# IPCA acumulado em 12 meses (proxy de inflação em 12m)
df["ipca_12m"] = df["ipca"].rolling(12).sum()

# Juros reais ex-post (SELIC - IPCA 12m)
df["selic_real"] = df["selic"] - df["ipca_12m"]

# Momentum de 3 meses (já usado no NB04, replicado aqui como feature)
df["ipca_momentum_3m"] = df["ipca"].rolling(3).mean()

# Spread entre SELIC e momentum
df["selic_minus_momentum"] = df["selic"] - df["ipca_momentum_3m"]

df.tail()

## 10. Dataset final para modelos preditivos

Montamos um dataframe limpo, pronto para ser usado em:
- regressão linear
- random forest
- XGBoost
- modelos híbridos (VAR + ML)

In [ ]:
cols_ml = [
    "ipca",
    "selic",
    "ipca_12m",
    "selic_real",
    "ipca_momentum_3m",
    "selic_minus_momentum",
] + [c for c in df.columns if "lag_" in c]

df_ml = df[cols_ml].dropna().copy()
df_ml.tail()

In [ ]:
# Salvar dataset de features para uso em outros notebooks
output_dir = os.path.join(ROOT, "data", "processed")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "ipca_selic_features.parquet")
df_ml.to_parquet(output_path)

print("Dataset de features salvo em:", output_path)

## 11. Análise de transmissão da política monetária

Pontos para interpretação:

- **Correlação com defasagem**:
  - Em que defasagem a correlação (negativa) entre SELIC e IPCA é mais forte?
  - Isso sugere um tempo típico de transmissão da política monetária.

- **Correlação móvel (rolling)**:
  - A relação entre juros e inflação é estável ao longo do tempo?
  - Há mudanças de regime (ex: períodos em que a política monetária foi menos eficaz)?

- **Gráfico SELIC defasada vs IPCA**:
  - Períodos de aperto monetário antecedem desaceleração da inflação?

- **IRF (VAR)**:
  - Um choque positivo na SELIC tende a reduzir o IPCA após alguns meses?
  - Qual a magnitude e o tempo de pico desse efeito?

- **FEVD**:
  - Qual a parcela da variância do IPCA explicada por choques na SELIC?

Este notebook fecha a ponte entre:
- estatística descritiva,
- econometria,
- e preparação de features para modelos de machine learning.